# AAV2 -- Plafond de bruit sur `viab` (attenuation correction + split-half empirique)

Pendant AAV2 d'`AAV5_viab_noise_ceiling.ipynb` (même session). Contrairement à AAV5 :
- comptages **entiers** (`compte_plasmide`/`compte_virus`, pas de granularité 0.5) -- le thinning
  binomial n'a donc aucun arrondi à gérer ;
- `compte_plasmide` plafonne à 61 (médiane=1) -- profondeur bien plus faible qu'AAV5 (500+) ;
- pas de bimodalité fit/non-fit détectée (`AAV2_viab_sorting.ipynb`, sweep de seuil + dithering) ;
- contamination virale diffuse (pas de spike-in ponctuel comme le 7m8 d'AAV5) -- **⚠ correction
  (2026-09-16)** : le cap `RATIO_MAX=100` qui gérait cette contamination a été retiré
  d'`AAV2_viab_sorting.ipynb` (il tronquait par construction toute la queue haute du log2
  enrichment réel, vérifié sur les données). `AAV2_organoides_sorted.csv` ne filtre donc plus DU
  TOUT cette contamination -- **caveat renforcé** : le plafond analytique/empirique ci-dessous ne
  modélise QUE le bruit de comptage Poisson, jamais cette contamination virale, qui est maintenant
  intégralement présente dans le CSV consommé ici (pas juste un résidu sous un seuil comme avant).

Mêmes deux estimations indépendantes qu'AAV5, contre la cible `viab` (`log2(virus/plasmide)`,
`AAV2_organoides_sorted.csv`, filtre `PLASMID_MIN=1` seul -- pas de cap sur le ratio) :

1. **Analytique** (delta-method Poisson) : `rho = 1 - var(bruit)/var(cible observée)`,
   plafond `sqrt(rho)`.
2. **Empirique** (thinning binomial + correction Spearman-Brown), sans hypothèse de forme.

Hors périmètre (comme pour AAV5) : loss Poisson/Multinomial -- ce notebook ne teste que le
plafond de bruit. Pas de dithering non plus (déjà écarté en session, ajoute juste du bruit).

**Pas exécuté automatiquement** -- léger (pandas/numpy sur un CSV déjà filtré), laissé pour que tu
le lances toi-même. **À relancer après avoir ré-exécuté `AAV2_viab_sorting.ipynb`** (le CSV
`AAV2_organoides_sorted.csv` doit être régénéré sans le cap ratio avant que les chiffres ici soient
valides).

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"   # avant tout import jax (via analysisV1)
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "Modelization_V2")
sys.path.insert(0, str(_root / "lib"))
LIB = _root / "lib"
from analysisV1 import AA_LABELS, pearson

CSV = Path("AAV2_organoides_sorted.csv")
if not CSV.exists():
    CSV = _root / "notebooks/notebooks/AAVs dataset/AAV2/AAV2_organoides_sorted.csv"
assert CSV.exists(), CSV

viab_col = "log2_enrichissement_virus_sur_plasmide"
use_cols = ["sequence", "compte_plasmide", "compte_virus", viab_col]
dtypes = {"sequence": "string", "compte_plasmide": "float32", "compte_virus": "float32", viab_col: "float32"}
df = pd.read_csv(CSV, usecols=use_cols, dtype=dtypes)
df["sequence"] = df["sequence"].astype("string")
print(df.shape)
df.head()


## 1. Cible recalculée (`eps=0.5`)

Convention projet : jamais `+1`. Sanity check contre la colonne fournie par le CSV (les comptages AAV2 sont des entiers purs -- contrairement à AAV5, pas de garantie a priori que le pseudocount de la colonne fournie soit déjà 0.5, donc ce check est d'autant plus utile ici).

In [ ]:
EPS = 0.5   # convention projet -- jamais +1 (cf. CLAUDE.md)

plasmid = df["compte_plasmide"].to_numpy(np.float64)
virus   = df["compte_virus"].to_numpy(np.float64)

target_ln   = np.log((virus + EPS) / (plasmid + EPS))   # base naturelle -- pour le calcul de plafond
target_log2 = target_ln / np.log(2)                       # base log2 -- pour comparer aux poids Potts déjà fittés

target_csv = df[viab_col].to_numpy(np.float64)
r_vs_csv = pearson(target_log2, target_csv)
print(f"n = {len(df):,}")
print(f"r(cible recalculée eps=0.5, colonne CSV fournie) = {r_vs_csv:+.4f}")


## 2. Plafond analytique -- correction d'atténuation (Poisson delta-method)

In [ ]:
# Var(ln X) ~= 1/lambda pour X ~ Poisson(lambda) (méthode delta) -- même forme que le poids
# inverse-variance déjà utilisé pour la régression de Potts : w = 1/(1/(n_num+eps)+1/(n_den+eps))
# est EXACTEMENT 1/noise_var sur le log ratio, en base naturelle.
noise_var_ln = 1.0 / (virus + EPS) + 1.0 / (plasmid + EPS)

mean_noise_var = noise_var_ln.mean()
observed_var   = target_ln.var()
rho            = 1.0 - mean_noise_var / observed_var
ceiling_r      = np.sqrt(max(rho, 0.0))

print(f"var(bruit de comptage), moyenne sur {len(df):,} variants : {mean_noise_var:.4f}")
print(f"var(cible observée)                                     : {observed_var:.4f}")
print(f"fiabilité rho = 1 - bruit/observé                       : {rho:.4f}")
print(f"plafond analytique sqrt(rho) (dénoiseur parfait vs cible observée) : {ceiling_r:.4f}")

noise_var_log2 = noise_var_ln / (np.log(2) ** 2)
rho_log2 = 1.0 - noise_var_log2.mean() / target_log2.var()
assert abs(rho_log2 - rho) < 1e-9, (rho_log2, rho)
print("(vérifié : rho identique en base log2, comme attendu -- rho est sans dimension)")


## 3. Plafond empirique -- thinning binomial + correction Spearman-Brown

Comptages entiers ici (contrairement à AAV5) : le split `Binomial(n, 0.5)` n'a aucun arrondi à gérer.

In [ ]:
def split_half_r(plasmid, virus, eps, rng):
    p_a = rng.binomial(plasmid.astype(np.int64), 0.5).astype(np.float64)
    p_b = plasmid - p_a
    v_a = rng.binomial(virus.astype(np.int64), 0.5).astype(np.float64)
    v_b = virus - v_a
    t_a = np.log((v_a + eps) / (p_a + eps))
    t_b = np.log((v_b + eps) / (p_b + eps))
    return pearson(t_a, t_b)

K = 30
rng = np.random.default_rng(0)
r_splits = np.array([split_half_r(plasmid, virus, EPS, rng) for _ in range(K)])
rho_half = r_splits.mean()

# Spearman-Brown (prophétie split-half) : chaque moitié a ~2x moins de comptage -> ~2x plus de
# bruit -> corr(moitié A, moitié B) sous-estime la fiabilité réelle à pleine profondeur.
rho_full_empirical = 2 * rho_half / (1 + rho_half)
ceiling_r_empirical = np.sqrt(max(rho_full_empirical, 0.0))

print(f"corr(moitié A, moitié B), moyenne sur {K} tirages : {rho_half:.4f} +/- {r_splits.std():.4f}")
print(f"fiabilité pleine profondeur (Spearman-Brown)      : {rho_full_empirical:.4f}")
print(f"plafond empirique sqrt(rho_full)                  : {ceiling_r_empirical:.4f}")
print(f"(à comparer au plafond analytique de la section 2 : {ceiling_r:.4f})")
print()
print("Si les deux plafonds divergent nettement (contrairement à AAV5 où ils devraient concorder) :")
print("signe possible que la contamination virale (plus filtrée du tout depuis le retrait du cap")
print("ratio, 2026-09-16) ajoute du bruit non-Poisson -- le split-half le capture (empirique,")
print("agnostique à la forme), pas la §2.")


## 4. Où se situe le fit Potts existant par rapport à ce plafond ?

`aav2_{F,J}_viab_potts_sorted_cv.npy` -- fit sur ce même CSV dans `AAV2_viab_sorting.ipynb` §9 (lambda CV=1e3, held-out r=+0.266, retenu contre lam=0 qui donnait +0.221). Deux points de référence supplémentaires, sur une population DIFFÉRENTE (top-10 000 variants par `compte_plasmide`, pas ce CSV) : Potts CV +0.252, ProfileMLP +0.274 (`AAV2_viab_top10k_potts_protocol_mlp.ipynb`).

In [ ]:
F_cv = np.load(LIB / "aav2_F_viab_potts_sorted_cv.npy")
J_cv = np.load(LIB / "aav2_J_viab_potts_sorted_cv.npy")

L, A = 7, 20
lut = np.zeros(256, dtype=np.int64)
for i, aa in enumerate(AA_LABELS):
    lut[ord(aa)] = i
seq_matrix = lut[np.frombuffer("".join(df["sequence"]).encode("ascii"), np.uint8)].reshape(len(df), L)

def score_FJ(seq, F, J):
    F, J = np.asarray(F), np.asarray(J)
    Fp = F[seq, np.arange(L)].sum(axis=1).astype(np.float64)
    Jp = np.zeros(len(seq), dtype=np.float64)
    for i in range(L):
        for j in range(i + 1, L):
            Jp += J[i, j, seq[:, i], seq[:, j]]
    return Fp + Jp

score_cv = score_FJ(seq_matrix, F_cv, J_cv)
r_insample_cv = pearson(score_cv, target_log2)

print("ATTENTION : ce r est calculé sur des variants qui ont en partie servi à fitter ces poids")
print("(in-sample, optimiste). Référence honnête = le r held-out déjà validé (+0.266).\n")

rows = [
    ("Potts (CV, ce CSV)         -- held-out validé", 0.266),
    ("Potts (CV, ce CSV)         -- in-sample (ce notebook)", r_insample_cv),
    ("Potts (CV, top10k plasmide) -- held-out, autre population", 0.252),
    ("ProfileMLP (top10k plasmide) -- held-out, autre population", 0.274),
    ("PLAFOND analytique sqrt(rho)  (section 2)", ceiling_r),
    ("PLAFOND empirique sqrt(rho) split-half Spearman-Brown  (section 3)", ceiling_r_empirical),
]
tbl = pd.DataFrame(rows, columns=["méthode", "r"]).set_index("méthode")
print(tbl.to_string(float_format=lambda x: f"{x:+.3f}"))

R_HELD_OUT = 0.266
best_ceiling = max(ceiling_r, ceiling_r_empirical)
frac = R_HELD_OUT / best_ceiling if best_ceiling > 0 else float("nan")
print(f"\nr held-out actuel / plafond estimé = {frac:.1%}")
if frac >= 0.85:
    print("-> le modèle est déjà proche du plafond de bruit : peu de marge, le problème est bien les données.")
elif frac >= 0.5:
    print("-> marge intermédiaire : une partie du signal extractible reste probablement sur la table.")
else:
    print("-> écart important au plafond : le modèle/la méthode de fit laisse du signal extractible de côté.")


## 5. Compromis profondeur de comptage / plafond -- où filtrer si on veut plus de signal ?

Seuil sur `compte_plasmide` seul (même variable que le `PLASMID_MIN_GRID` d'`AAV2_viab_sorting.ipynb` -- le plasmide est le facteur limitant identifié là-bas, pas le virus) sur ce CSV filtré `PLASMID_MIN=1` seul (pas de cap sur le ratio depuis le 2026-09-16).

In [ ]:
def ceiling_at_threshold(min_plasmid):
    mask = plasmid > min_plasmid
    n = int(mask.sum())
    if n < 100:
        return n, np.nan, np.nan
    t = target_ln[mask]
    nv = noise_var_ln[mask]
    rho_t = 1.0 - nv.mean() / t.var()
    return n, rho_t, np.sqrt(max(rho_t, 0.0))

thresholds = [0, 1, 2, 3, 5, 8, 10, 15, 20, 30, 40, 50]   # même grille qu'AAV2_viab_sorting.ipynb §PLASMID_MIN_GRID (+30/40/50)
rows = [(t, *ceiling_at_threshold(t)) for t in thresholds]
sweep = pd.DataFrame(rows, columns=["compte_plasmide >", "n_variants", "rho", "plafond sqrt(rho)"])

fig, ax1 = plt.subplots(figsize=(7.5, 4.5))
ax2 = ax1.twinx()
ax1.plot(sweep["compte_plasmide >"], sweep["plafond sqrt(rho)"], "o-", color="crimson", label="plafond sqrt(rho)")
ax2.plot(sweep["compte_plasmide >"], sweep["n_variants"], "s--", color="steelblue", label="n variants")
ax2.set_yscale("log")
ax1.set_xlabel("seuil compte_plasmide >")
ax1.set_ylabel("plafond de corrélation sqrt(rho)", color="crimson")
ax2.set_ylabel("variants retenus (log)", color="steelblue")
ax1.set_title("Compromis profondeur de comptage / plafond de bruit -- viab AAV2")
fig.tight_layout()
plt.show()
sweep


## Lecture

- `compte_plasmide` plafonne à 61 pour AAV2 (contre 500+ pour AAV5) : même au seuil le plus haut
  testable ici, la profondeur reste modeste -- si le plafond §5 continue de monter sans saturer sur
  toute la grille, ça confirme que le plasmide (pas le virus, largement mieux mesuré ici) est le
  vrai goulot d'étranglement de l'information, cohérent avec le diagnostic déjà fait dans
  `AAV2_viab_sorting.ipynb`.
- Si le r held-out actuel (+0.266, PÉRIMÉ -- calculé avant le retrait du cap ratio) est déjà proche
  des plafonds §2/§3 : données limitantes, la piste la plus rentable est de pousser plus loin le
  filtrage par profondeur (comme `top10k`/`top50k` l'explorent déjà) plutôt qu'un modèle plus gros.
- Si les plafonds §2 (Poisson pur) et §3 (empirique, agnostique) divergent nettement l'un de
  l'autre : signe qu'une contamination résiduelle ajoute du bruit non-Poisson -- **cette
  contamination n'est plus du tout filtrée depuis le retrait du cap `RATIO_MAX=100` (2026-09-16)**,
  donc cette divergence pourrait être plus marquée qu'avant -- le plafond §3 est alors la lecture
  la plus fiable des deux.